# BuildingMCP Servers

**Module 10 · Lesson 10.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Your First MCP Server
- The Three Primitives
- Transports: stdio vs Streamable HTTP
- Connect a Client
- Use It in Claude Desktop (and Cursor, VS Code)
- Test & Debug

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## From a Function to a Tool Any AI Can Use

> 💡 **Info**
>
> You have a plain Python function: `def get_gst(amount): return round(amount * 1.18, 2)`. Right now only your code can call it. Add two things - the decorator `@mcp.tool()` above it and `mcp.run()` at the bottom - and it becomes an **MCP server**. Now Claude Desktop can call it, Cursor can call it, VS Code can call it, and so can any other MCP client, because they all speak the same protocol (10.4). You did not write a schema, a JSON-RPC handler, or a transport - FastMCP generated all of it from your type hints and docstring. That is the whole promise of building an MCP server: **write the function once, and every AI client can use it.**

### 🎯 What you will be able to do after this lesson

- **Build** an MCP server with FastMCP - `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`, `mcp.run()` - with the schema generated from your type hints.

- **Compare** the two transports (stdio subprocess vs Streamable HTTP remote) and the three primitives (tools DO, resources READ, prompts TEMPLATE), and connect a client through the JSON-RPC handshake (initialize → list → call).

- **Operate** the dev loop: configure the server in Claude Desktop / Cursor / VS Code, and test + debug with `fastmcp dev` (the MCP Inspector) and the in-memory `Client`.

- **Evaluate** how to compose + scale (FastMCP 3.0 mount / proxy / OpenAPI-to-MCP) and deploy stdio → Streamable HTTP on Cloud Run - including India’s MoSPI eSankhyiki, Sarvam, and Bhashini MCP servers.

> 📦 **Info**
>
> ✅ Before you startThis builds on **Lesson 10.4** (what MCP is: one server, any client, N+M) and 10.1–10.2 (the tool-use loop and schema design). You should know what a tool schema is and that MCP standardizes tool exposure over JSON-RPC 2.0. Comfort with Python decorators and basic `async`/`await` helps.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍴 **Analogy**
>
> **An MCP server is a restaurant that publishes a standard menu.** The menu format is the protocol (10.4). **Tools** are dishes you can order (actions); **resources** are the ingredients list you can read; **prompts** are the chef’s set-menus. Any diner who can read the standard menu - Claude, Cursor, VS Code - can order without a custom deal with the kitchen. You cook once; every client orders. **Where it breaks down:** a real kitchen trusts its diners; an MCP server must not - it validates every order against the schema and guards effectful dishes (10.1/10.2 carry over).

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“An MCP server is a big framework.”** A minimal server is a `FastMCP` object + one decorated function + `mcp.run()`. The framework hides the JSON-RPC and transport boilerplate.
> - **“I have to write the schema.”** `@mcp.tool()` builds it from your type hints and docstring. Add hints and a docstring; that IS the schema.

> 💡 **Info**
>
> ⚠️ Anti-patternModeling a read-only lookup (a catalog, a config, a DB row) as a **tool** instead of a **resource**. It works, but you lose the resource semantics hosts rely on (caching, addressability, “this is safe to read”). Tools DO; resources READ - pick the right primitive.

---

## 🎯 Concept 1: Your First MCP Server

### Your First MCP Server

A FastMCP object, a decorated function, and mcp.run(). The schema is generated for you.

A server is astonishingly small. Create a `FastMCP` object, decorate a function with `@mcp.tool()`, and call `mcp.run()`. That is a complete, discoverable MCP server. FastMCP reads your **type hints and docstring** and generates the tool’s input schema (the 10.2 schema, for free); it wires up JSON-RPC and the transport. `@mcp.resource("uri://...")` adds read-only data; `@mcp.prompt()` adds a template. Use the standalone `fastmcp` package (FastMCP 3.x, which powers most MCP servers); the raw MCP SDK is the low-level escape hatch for custom transports.

> 📜 **Analogy**
>
> It’s **publishing a recipe to a standard cookbook**. You write the dish once (the function); the cookbook’s standard format (the decorator) makes it readable by every kitchen that owns the cookbook (every MCP client). You don’t translate the recipe for each kitchen - the format does that.

You add `@mcp.tool()` to a typed, docstringed function and run `mcp.run()`. What must you write by hand?

**📝 `01_server.py FastMCP`** - *3.x*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
!pip install python-dotenv fastmcp -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


- `FastMCP("Netsetos")` is the server; each decorator registers one primitive.
- `@mcp.tool()` reads `add_gst(amount: float)` + the docstring and generates the input schema - you never write JSON Schema.
- `@mcp.resource("catalog://courses")` exposes read-only data at a URI; `@mcp.prompt()` a reusable template.
- `mcp.run()` starts it on stdio - a client can now discover all three. Illustrative: it needs `pip install fastmcp`.

#### 💡 What just happened

⚡ What just happened? A server is a `FastMCP` object + decorated functions + `mcp.run()`. The decorator generates the schema from your hints and docstring and hides the JSON-RPC + transport boilerplate. The rest of this lesson is: which primitive, which transport, how a client connects, and how to test and ship it. Tradeoff: FastMCP hides the boilerplate; drop to the raw MCP SDK instead only when you need a custom transport or non-standard messages.

- Toggle type hints and the docstring
- Watch the generated tool schema and the server expose it

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Three Primitives

### The Three Primitives

Tools DO (actions), Resources READ (addressable data), Prompts TEMPLATE (reusable messages). Pick the right one.

MCP has exactly three primitives, and choosing correctly matters. A **tool** is a function the model invokes to DO something (side effects allowed). A **resource** is addressable read-only data the client GETs by URI - a file, a config, a catalog; think of it like an HTTP GET. A **prompt** is a reusable message template the user or host can invoke. Modeling a read as a tool works, but it discards the resource semantics hosts depend on. In FastMCP these are just three decorators over your functions.

> 🎪 **Analogy**
>
> It’s a **library**. **Tools** are the services the librarian performs for you (reserve a book, pay a fine) - actions. **Resources** are the books on the shelf you simply read - addressable, read-only. **Prompts** are the pre-printed request forms - templates you fill in. You don’t ask the librarian to ‘perform’ a book at you; you read it.

You expose a read-only course catalog fetched by URI. Which primitive?

**📝 `02_primitives.py`** - *runnable*

In [ ]:
# THE THREE PRIMITIVES - a tiny stand-in for FastMCP to show the SHAPE (tools DO, resources READ, prompts TEMPLATE).
class MiniMCP:
    def __init__(self, name): self.name=name; self.tools={}; self.resources={}; self.prompts={}
    def tool(self):
        def deco(fn): self.tools[fn.__name__] = fn; return fn
        return deco
    def resource(self, uri):
        def deco(fn): self.resources[uri] = fn; return fn
        return deco
    def prompt(self):
        def deco(fn): self.prompts[fn.__name__] = fn; return fn
        return deco

mcp = MiniMCP("Netsetos")

@mcp.tool()
def add_gst(amount):                                # a TOOL: an action the model invokes
    return round(amount * 1.18, 2)
@mcp.resource("catalog://courses")
def courses():                                      # a RESOURCE: read-only data the client GETs
    return [{"id": "genai-bootcamp", "price_inr": 14999}]
@mcp.prompt()
def quote(course):                                  # a PROMPT: a reusable template
    return f"Price of {course} with GST, one line."

print("  Server 'Netsetos' exposes:")
print(f"    tools     = {list(mcp.tools)}       (DO: invoke an action)")
print(f"    resources = {list(mcp.resources)}   (READ: GET addressable data)")
print(f"    prompts   = {list(mcp.prompts)}          (TEMPLATE: reusable message)")
print(f"  client calls a tool: add_gst(14999) -> {mcp.tools['add_gst'](14999)}")

# Output:
#   Server 'Netsetos' exposes:
#     tools     = ['add_gst']       (DO: invoke an action)
#     resources = ['catalog://courses']   (READ: GET addressable data)
#     prompts   = ['quote']          (TEMPLATE: reusable message)
#   client calls a tool: add_gst(14999) -> 17698.82

- `MiniMCP` is a tiny stand-in for FastMCP - the point is the SHAPE, not the real library.
- Three decorators register into three buckets: `tools`, `resources`, `prompts`.
- The client lists each bucket and calls a tool - exactly what a real MCP client does over JSON-RPC (Step 4).
- When to use which: DO -> tool, READ -> resource, TEMPLATE -> prompt. Mismatching them is the anti-pattern above.

Most real tools are **effectful** - they write a row, send a message, or move money. MCP lets you annotate that intent with **tool hints** (`readOnlyHint`, `destructiveHint`, `idempotentHint`) so the host can decide the UX: auto-run a safe read, but ask the user to confirm a destructive action. Hints are advisory, not enforcement - you still validate inputs and guard effectful tools yourself.

**📝 `02b_effectful.py`** - *illustrative*

In [ ]:
# EFFECTFUL TOOLS - most real tools DO something (write a row, move money). Annotate the intent.
from fastmcp import FastMCP
mcp = FastMCP("Netsetos")

@mcp.tool(annotations={"readOnlyHint": True})       # safe: only reads, no side effects
def get_invoice(invoice_id: str) -> dict:
    "Look up an invoice."
    return {"id": invoice_id, "status": "paid"}

@mcp.tool(annotations={"destructiveHint": True, "idempotentHint": False})  # mutates; unsafe to retry
def refund_invoice(invoice_id: str) -> dict:
    "Refund an invoice - a real side effect (moves money)."
    # ... perform the refund ...
    return {"id": invoice_id, "refunded": True}

# Output: (illustrative) annotations are advisory HINTS the host reads to shape the UX -
#         it may auto-run a readOnlyHint tool but ASK the user to confirm a destructiveHint one.
#         Hints are not enforcement: still validate inputs and guard effectful tools yourself.

#### 💡 What just happened

⚡ What just happened? Three primitives, three decorators. **Tools** DO, **resources** READ, **prompts** are TEMPLATES. A server exposes any mix. Picking the right primitive gives hosts the semantics they optimize for (a resource is cacheable and safe-to-read; a tool may have side effects).

- Classify a use-case as tool / resource / prompt
- See the semantics each one gives the host

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Transports: stdio vs Streamable HTTP

### Transports: stdio vs Streamable HTTP

The same server runs two ways. Local = stdio subprocess. Remote = Streamable HTTP. A re-run, not a rewrite.

Your server code does not change between local and remote - only how it is *transported*. **stdio** (the default) means the client launches your server as a *subprocess* and talks over stdin/stdout; this is how desktop hosts (Claude Desktop, Cursor, VS Code) run a LOCAL server. **Streamable HTTP** - `mcp.run(transport="streamable-http", host="0.0.0.0", port=...)` - is a network server any remote client connects to; it replaced the older HTTP+SSE transport (2025-03-26 spec revision). Same functions, same schemas; one argument to `mcp.run` flips it.

> 🔌 **Analogy**
>
> It’s a **power tool with two plugs**. Indoors you plug it into the wall socket right next to you (stdio - a direct local connection). On a job site you run it off the site’s generator over a long cable (Streamable HTTP - reachable from anywhere). Same tool; you just choose the plug.

Your stdio server works locally. To reach remote clients, what do you change?

**📝 `03_transports.py`** - *runnable*

In [ ]:
# TRANSPORTS - the SAME server runs two ways; you change ONE argument, not the code.
import os
def run(transport="stdio", host="0.0.0.0", port=None):
    if transport == "stdio":
        return "stdio: client launches this file as a SUBPROCESS; talk over stdin/stdout (local hosts)"
    if transport == "streamable-http":
        port = port or int(os.environ.get("PORT", 8000))
        return f"streamable-http: serving on http://{host}:{port} (remote clients; replaced SSE)"
    raise ValueError("unknown transport")

print("  Local dev  -> mcp.run()")
print("    ", run("stdio"))
print("  Production -> mcp.run(transport='streamable-http', host='0.0.0.0', port=int(os.environ['PORT']))")
print("    ", run("streamable-http", port=8080))

# Output:
#   Local dev  -> mcp.run()
#      stdio: client launches this file as a SUBPROCESS; talk over stdin/stdout (local hosts)
#   Production -> mcp.run(transport='streamable-http', host='0.0.0.0', port=int(os.environ['PORT']))
#      streamable-http: serving on http://0.0.0.0:8080 (remote clients; replaced SSE)

- The SAME server object is run two ways; `run()` here just names what each transport means.
- `stdio`: the client spawns your file as a subprocess and pipes JSON-RPC over stdin/stdout - local only.
- `streamable-http`: a real network endpoint on `0.0.0.0:$PORT` - remote clients connect; it replaced SSE.
- Tradeoff: stdio is zero-config for local hosts but not reachable remotely; Streamable HTTP is reachable but needs hosting (Step 8).

#### 💡 What just happened

⚡ What just happened? One server, two transports. **stdio** for local hosts (they spawn a subprocess); **Streamable HTTP** for remote clients. Switching is a change to `mcp.run`’s argument, not a rewrite - the same functions and schemas ride either transport. Streamable HTTP replaced SSE; do not reach for SSE.

- Flip the transport on the SAME server
- Watch it run as a local subprocess vs a remote endpoint

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Connect a Client

### Connect a Client

A client speaks JSON-RPC 2.0: initialize, then discover (list), then call. FastMCP’s Client does it in three lines.

Everything a client does is **JSON-RPC 2.0**, and the lifecycle is always the same: **initialize** (a version + capabilities handshake), then **list** (discover the tools, resources, and prompts), then **call**. A host like Claude does this for you; when you write a client yourself, FastMCP’s async `Client` is three lines: `async with Client(server) as c: await c.list_tools(); await c.call_tool("add_gst", {"amount": 14999})`. The `Client` can point at a running server (a file path or URL) or, for tests, directly at a server instance in memory (Step 6).

> 🤝 **Analogy**
>
> It’s **meeting someone new at a conference**. First you introduce yourselves and confirm you speak the same language (initialize). Then you ask what they do (list - discover their tools). Only then do you make a specific request (call). You don’t ask for a favor before you’ve even said hello.

What is the correct order of a client’s JSON-RPC calls to an MCP server?

**📝 `04_client.py`** - *runnable*

In [ ]:
# THE CLIENT + JSON-RPC 2.0 HANDSHAKE - every MCP client: initialize -> discover -> call.
def rpc(method, params=None, _id=[0]):
    _id[0] += 1
    return {"jsonrpc": "2.0", "id": _id[0], "method": method, "params": params or {}}

handshake = [
    rpc("initialize", {"protocolVersion": "2025-11-25", "capabilities": {}}),
    rpc("tools/list"),
    rpc("tools/call", {"name": "add_gst", "arguments": {"amount": 14999}}),
]
labels = ["handshake (version + capabilities)", "discover the tools", "invoke a tool"]
print("  The three JSON-RPC messages every MCP client sends:")
for msg, lab in zip(handshake, labels):
    print(f"    {msg['method']:12s} (id={msg['id']}) - {lab}")
resp = {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "17698.82"}]}}
print(f"  server replies -> result.content[0].text = {resp['result']['content'][0]['text']}")

# With FastMCP the real client is 3 lines (async):
#   from fastmcp import Client
#   async with Client("server.py") as c:      # or Client(mcp) in-memory
#       await c.call_tool("add_gst", {"amount": 14999})

# Output:
#   The three JSON-RPC messages every MCP client sends:
#     initialize   (id=1) - handshake (version + capabilities)
#     tools/list   (id=2) - discover the tools
#     tools/call   (id=3) - invoke a tool
#   server replies -> result.content[0].text = 17698.82

- `rpc()` builds a JSON-RPC 2.0 message (jsonrpc + id + method + params) - the wire format MCP uses.
- The three methods are the whole lifecycle: `initialize` (handshake), `tools/list` (discover), `tools/call` (invoke).
- The server’s reply nests the answer under `result.content[]` - MCP results are typed content blocks.
- The real FastMCP `Client` (commented) does all of this in three async lines; here we show the messages so the protocol is tangible.

#### 💡 What just happened

⚡ What just happened? A client is: **initialize → list → call**, all JSON-RPC 2.0. Hosts (Claude, Cursor, VS Code) do it for you; FastMCP’s `Client` does it in three lines when you need your own. When to use which: let the host handle it for interactive use, and write your own `Client` only for tests or custom automation. The message shapes here are exactly what flows on the wire, whichever transport carries them.

- Step initialize -> list -> call
- See the real JSON-RPC message at each stage

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Use It in Claude Desktop (and Cursor, VS Code)

### Use It in Claude Desktop (and Cursor, VS Code)

Point the host at your server with claude_desktop_config.json, or one fastmcp command.

To use your server in a desktop host, tell the host how to launch it. For Claude Desktop that is **`claude_desktop_config.json`** - a JSON object with an `mcpServers` key mapping a name to the `command` + `args` (and optional `env`) the host runs to spawn your stdio server. Cursor and VS Code use the same `mcpServers` shape. The shortcut is `fastmcp install claude-desktop server.py`, which writes the config and manages dependencies for you. Restart the host and your tools appear.

After adding your server to `claude_desktop_config.json`, it doesn’t show up. Most likely fix?

**📝 `05_config.py`** - *runnable*

In [ ]:
# HOST CONFIG - point Claude Desktop (Cursor / VS Code) at your server via claude_desktop_config.json.
import json
def desktop_config(name, command, args, env=None):
    entry = {"command": command, "args": args}
    if env: entry["env"] = env
    return {"mcpServers": {name: entry}}

cfg = desktop_config("netsetos", "python", ["-m", "netsetos_server"],
                     env={"NETSETOS_API_KEY": "..."})
print("  claude_desktop_config.json (the host spawns your stdio server):")
print("   ", json.dumps(cfg, separators=(",", ":")))
srv = cfg["mcpServers"]["netsetos"]
ok = bool(srv.get("command")) and isinstance(srv.get("args"), list)
print(f"  valid launch spec? {ok}  (command + args -> host can spawn it; restart the host to load)")
print("  Or run: fastmcp install claude-desktop server.py   (writes this config for you)")

# Output:
#   claude_desktop_config.json (the host spawns your stdio server):
#     {"mcpServers":{"netsetos":{"command":"python","args":["-m","netsetos_server"],"env":{"NETSETOS_API_KEY":"..."}}}}
#   valid launch spec? True  (command + args -> host can spawn it; restart the host to load)
#   Or run: fastmcp install claude-desktop server.py   (writes this config for you)

- `desktop_config` builds the `mcpServers` entry: a `command` + `args` the host runs to spawn your server.
- The validation checks the launch spec has what the host needs (command + args) - a missing one means the server never starts.
- `env` passes secrets/config into the subprocess (never hard-code keys in the server).
- `fastmcp install claude-desktop` writes this file for you; Cursor and VS Code use the same shape. Restart the host to load it.

#### 💡 What just happened

⚡ What just happened? A desktop host runs your LOCAL server over stdio, so it needs a launch spec: `command` + `args` in `claude_desktop_config.json` (or `fastmcp install claude-desktop`). Same shape for Cursor and VS Code. Restart the host after editing - it reads the config at startup. Tradeoff: a local stdio server is configured by `command` + `args`; a remote server (Step 8) is configured by URL instead.

---

## 🎯 Concept 6: Test & Debug

### Test & Debug

In-memory Client for pytest (no subprocess); fastmcp dev launches the MCP Inspector for interactive debugging.

You do not test an MCP server by clicking around in Claude. Two better tools. For automated tests, FastMCP’s **in-memory `Client`** connects a real client directly to your server instance over `FastMCPTransport` - no subprocess, no network - so `await client.call_tool(...)` runs in-process and is fully assertable in pytest. For interactive debugging, **`fastmcp dev`** launches the **MCP Inspector** (also `npx @modelcontextprotocol/inspector`): a browser UI that lists your tools/resources/prompts, calls them live, and shows the raw JSON-RPC. And remember: the input schema is generated from your signature, so a bad hint is a bad schema - test it.

How do you unit-test an MCP tool in CI without spinning up a subprocess or network?

**📝 `06_test.py`** - *runnable*

In [ ]:
# TEST + DEBUG - @mcp.tool() builds the schema from your signature; test with an in-memory client.
import inspect, json
JSON = {int: "integer", float: "number", str: "string", bool: "boolean"}
def schema_from(fn):
    props, required = {}, []
    for pname, p in inspect.signature(fn).parameters.items():
        props[pname] = {"type": JSON.get(p.annotation, "string")}
        if p.default is inspect._empty: required.append(pname)
    return {"type": "object", "properties": props, "required": required,
            "description": (fn.__doc__ or "").strip()}

def add_gst(amount: float) -> float:
    "Add 18% GST to a rupee amount."
    return round(amount * 1.18, 2)

print("  @mcp.tool() generates this inputSchema from add_gst(amount: float):")
print("   ", json.dumps(schema_from(add_gst), separators=(",", ":")))
got = add_gst(100)                                  # an in-memory client call: no subprocess, no network
assert got == 118.0, got
print(f"  in-memory test: call_tool('add_gst', amount=100) -> {got}  (assert == 118.0 PASSED)")

# Output:
#   @mcp.tool() generates this inputSchema from add_gst(amount: float):
#     {"type":"object","properties":{"amount":{"type":"number"}},"required":["amount"],"description":"Add 18% GST to a rupee amount."}
#   in-memory test: call_tool('add_gst', amount=100) -> 118.0  (assert == 118.0 PASSED)

- `schema_from` mirrors what `@mcp.tool()` does - it reads the signature and builds the `inputSchema`.
- So a wrong type hint gives a wrong schema; testing the tool catches it before a client does.
- The in-memory call is a direct invocation - the real FastMCP `Client(server)` does the same over `FastMCPTransport`, with no subprocess.
- Debug interactively with `fastmcp dev` (the MCP Inspector) to see live calls and raw JSON-RPC.

One more essential: **what happens when a tool fails?** Validate inputs and `raise ToolError(...)` so the client gets a clean, intentional error (a tools/call result flagged `isError`) instead of a server traceback - and the model can read the message and retry. FastMCP also catches unhandled exceptions and returns them as errors, but an explicit `ToolError` is the message you control.

**📝 `06b_errors.py`** - *illustrative*

In [ ]:
# WHEN A TOOL RAISES - validate inputs and raise ToolError; the client gets a clean error, not a crash.
from fastmcp import FastMCP
from fastmcp.exceptions import ToolError

mcp = FastMCP("Netsetos")

@mcp.tool()
def add_gst(amount: float) -> float:
    "Add 18% GST to a rupee amount (amount must be non-negative)."
    if amount < 0:
        raise ToolError("amount must be >= 0")   # a controlled, client-visible error
    return round(amount * 1.18, 2)

# Output: (illustrative) add_gst(-5) does NOT crash the server. FastMCP catches the ToolError and
#         returns a tools/call RESULT with isError=true carrying your message:
#         {"content":[{"type":"text","text":"amount must be >= 0"}],"isError":true}
#         The model SEES that message and can retry with a valid argument. A raised ToolError is
#         reported to the client; an unhandled exception is caught too, but its detail is masked.

#### 💡 What just happened

⚡ What just happened? Test with the **in-memory `Client`** (a real client, no subprocess/network - assertable in pytest); debug with **`fastmcp dev`** (the MCP Inspector). Because the schema is generated from your signature, testing the tool also tests the schema. When to use which: the in-memory `Client` for fast, assertable CI tests; the Inspector for eyeballing a live server by hand. And a tool that validates its inputs and raises `ToolError` fails cleanly (a client-visible error the model can recover from), not with a traceback. This is the dev loop that makes server-building fast and safe.

---

## 🎯 Concept 7: Compose & Scale

### Compose & Scale

FastMCP 3.0 composes servers: mount a local one, proxy a remote one, or turn a REST API into MCP. Namespaces resolve clashes.

Real deployments combine servers. FastMCP 3.0 frames a server as **Providers** (where components come from - decorators, a directory, an OpenAPI spec, or a remote server) plus **Transforms** (middleware that renames, namespaces, filters, or secures a provider without touching its code). Concretely: `mcp.mount(sub, prefix="billing")` composes a LOCAL sub-server; `FastMCP.as_proxy(url)` exposes a REMOTE server’s tools as if local; `FastMCP.from_openapi(spec)` turns a whole REST API into MCP tools. On a name clash, prefixing keeps them apart and the most-recently-mounted server wins. Auth plugs in via the `TokenVerifier` protocol.

> 🏪 **Analogy**
>
> It’s a **food court**. Each stall is its own kitchen (a server); the food court (the composed server) gives diners one place to order from all of them. If two stalls both sell ‘noodles,’ you label them by stall (a prefix) so orders don’t collide. A franchise stall piping in from another location is a proxy.

Two mounted servers both define `forecast`. What prevents a clash?

**📝 `07_compose.py`** - *runnable*

In [ ]:
# COMPOSE - FastMCP 3.0 merges servers with mount(prefix=...); on a name clash, the last mount wins.
def mount(base, sub, prefix):
    for name, fn in sub.items():
        base[f"{prefix}_{name}"] = fn                # prefixing avoids clashes
    return base

weather = {"forecast": lambda: "34C"}
billing = {"forecast": lambda: "invoice #42"}        # clashes with weather on 'forecast'
combined = {}
mount(combined, weather, "weather")
mount(combined, billing, "billing")
print(f"  Two sub-servers mounted -> one tool surface: {sorted(combined)}")

proxy = dict(combined)
proxy["remote_search"] = lambda: "(proxied from a remote MCP server)"   # FastMCP.as_proxy()
print(f"  FastMCP.as_proxy() adds a REMOTE server's tools as if local -> {len(proxy)} tools total")

# The real FastMCP calls:
#   main.mount(billing_server, prefix="billing")     # compose a LOCAL sub-server
#   proxy = FastMCP.as_proxy("https://remote/mcp")   # expose a REMOTE server as if local
#   FastMCP.from_openapi(spec, client)               # turn a whole REST API into MCP tools

# Output:
#   Two sub-servers mounted -> one tool surface: ['billing_forecast', 'weather_forecast']
#   FastMCP.as_proxy() adds a REMOTE server's tools as if local -> 3 tools total

- `mount` prefixes each sub-server’s components (`weather_forecast`, `billing_forecast`) so both coexist.
- `FastMCP.as_proxy(url)` adds a REMOTE server’s tools to the same surface as if they were local - the client sees one server.
- The real calls (commented) are `mount(prefix=...)`, `FastMCP.as_proxy(url)`, and `FastMCP.from_openapi(spec)`.
- Tradeoff: composition adds surface area and a coordination point; namespace deliberately so names stay unambiguous.

#### 💡 What just happened

⚡ What just happened? Compose servers, don’t rewrite them. **mount** a local sub-server (prefix to namespace; last-wins on a clash), **proxy** a remote one, or turn a whole **OpenAPI** REST API into MCP tools. FastMCP 3.0’s Providers + Transforms make this a few lines. Auth attaches via `TokenVerifier`; the deep production hardening is Lesson 10.7.

- Mount a local server and proxy a remote one
- Force a name clash and watch last-wins resolve it

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: Deploy: stdio → Cloud Run (and India)

### Deploy: stdio → Cloud Run (and India)

Local dev is stdio; production is a Streamable-HTTP container on Cloud Run. Plus real Indian MCP servers.

Shipping is the same server with the remote transport, containerized. Locally you run `mcp.run()` (stdio). For production you run `mcp.run(transport="streamable-http", host="0.0.0.0", port=int(os.environ["PORT"]))` inside a Docker image and deploy it to **Cloud Run**, which scales your server across client connections. Bind `0.0.0.0`, read the injected `$PORT`, and add a health check; watch for the Cloud Run “Invalid Host Header’’ / 421 gotcha. **India:** the **MoSPI eSankhyiki** MCP server (official, MIT-licensed, launched 2026) already exposes seven government statistics datasets over MCP; **Sarvam** and **Bhashini** are naturally wrapped as MCP servers so Indic apps discover them like any other tool. The deep remote-deploy path is Lesson 10.6.

> 🏭 **Analogy**
>
> It’s the difference between **running an appliance on the workbench next to you and renting it a bay in a serviced workshop**. Local **stdio** is the bench - right there, one user, zero setup. **Cloud Run** is the serviced workshop - you hand over a sealed crate (the container image), and it gives your server a public address, keeps it running, and lets many clients use it at once.

Deploying a production MCP server to Cloud Run. What is current and correct?

**📝 `08_deploy.py`** - *runnable*

In [ ]:
# DEPLOY - local stdio for dev; Streamable HTTP in a container on Cloud Run for production.
def deploy_target(where):
    if where == "local":
        return ("stdio", "client spawns a subprocess; no network")
    if where == "cloud-run":
        return ("streamable-http", "container binds 0.0.0.0:$PORT; Cloud Run fans out across clients")
    raise ValueError("unknown target")

for where in ["local", "cloud-run"]:
    transport, note = deploy_target(where)
    print(f"  {where:9s} -> transport={transport:15s} | {note}")

# Dockerfile (essentials):
#   FROM python:3.12-slim
#   RUN pip install fastmcp
#   COPY server.py .
#   CMD ["python", "server.py"]   # server.py: mcp.run(transport="streamable-http",
#                                 #             host="0.0.0.0", port=int(os.environ["PORT"]))

# Output:
#   local     -> transport=stdio           | client spawns a subprocess; no network
#   cloud-run -> transport=streamable-http | container binds 0.0.0.0:$PORT; Cloud Run fans out across clients

- `deploy_target` maps the environment to the transport: local = stdio, Cloud Run = streamable-http.
- The Dockerfile (commented) is minimal: install fastmcp, copy the server, and run it binding `0.0.0.0:$PORT`.
- Cloud Run injects `$PORT` and load-balances client connections across instances - read the env var, do not hard-code.
- India: MoSPI eSankhyiki is a real government MCP server (7 datasets); Sarvam/Bhashini wrap as MCP so Indic tools are discoverable.

> 📦 **Info**
>
> 🔐 Public by defaultA Streamable-HTTP server on Cloud Run is reachable by **anyone who learns the URL** - FastMCP adds no authentication by default. Before you expose real tools or data, put auth in front of it: attach a `TokenVerifier` (Step 7), require an auth token, and restrict who can reach the endpoint. Never deploy an unauthenticated server that can DO something. The full hardening path (auth depth, rate limits) is Lesson 10.7.

#### 💡 What just happened

⚡ What just happened? Deploy = the same server on the remote transport, containerized. Local **stdio** for dev; a **Streamable-HTTP** container on **Cloud Run** (bind `0.0.0.0:$PORT`, health check) for production. India already has real MCP servers (MoSPI eSankhyiki, plus Sarvam/Bhashini). Tradeoff: stdio is simplest for local dev but not reachable remotely; a Streamable-HTTP container is reachable but needs hosting. The deep remote-deploy walkthrough is next, in Lesson 10.6.

- Step local stdio -> container -> Cloud Run
- Watch the remote endpoint fan out across clients

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Building an MCP server is small and layered. A `FastMCP` object plus decorated functions IS the server, with the schema generated for you (Step 1). Choose the right primitive - tools DO, resources READ, prompts TEMPLATE (Step 2). The same server runs local (stdio) or remote (Streamable HTTP) by one argument (Step 3), and any client reaches it through initialize → list → call (Step 4). Wire it into Claude Desktop / Cursor / VS Code (Step 5), test it with the in-memory `Client` and debug it in the MCP Inspector (Step 6), compose servers with mount / proxy / OpenAPI (Step 7), and deploy stdio → Streamable HTTP on Cloud Run (Step 8). Write the function once; every AI client can use it.

> 📦 **Info**
>
> ➡️ Where this goes nextNext up, in Lesson 10.6 you take the Streamable-HTTP server from Step 8 and do the full remote deploy on Cloud Run - networking, scaling, and secure access. Production hardening (auth depth, rate limits, versioning, observability) arrives in Lesson 10.7. The reference is the [FastMCP documentation](https://gofastmcp.com/getting-started/welcome) and the [MoSPI eSankhyiki MCP server](https://github.com/nso-india/esankhyiki-mcp).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **BuildingMCP Servers**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-10_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-10.5.html` - regenerate this notebook whenever the source HTML is updated.*
